In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS ecommerce_bronze
COMMENT 'Bronze layer tables for Olist e-commerce project'
""")


In [0]:
from pyspark.sql.functions import *

csv_base_path = '/Volumes/workspace/e-commerce/e-commerce/'
csv_files = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv"
]

for file_name in csv_files:
    df = spark.read.option('header', True).option('inferSchema', True).csv(f"{csv_base_path}{file_name}")
    print(f"\nSchema for {file_name}")
    df.printSchema()

    table_name = file_name.replace("olist_", "").replace("_dataset.csv","").replace(".csv","")

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"ecommerce_bronze.{table_name}")

    
    print(f" {file_name} written as Unit Catalog table ecommerce_bronze.{table_name}")

    
    

In [0]:
%sql
select * from ecommerce_bronze.orders limit 10

In [0]:
%sql
select * from ecommerce_bronze.order_items limit 10

In [0]:
%sql
select * from ecommerce_bronze.products limit 10

In [0]:
%sql
select * from ecommerce_bronze.geolocation limit 10

In [0]:
from pyspark.sql.functions import col

spark.sql("create database if not exists ecommerce_silver")
silver_rules ={
    'customers':["customers_id"],
    'orders':["order_id"],
    'order_items':["order_id","product_id"],
    'order_payments':["order_id"],
    'order_reviews':["review_id"],
    'products':['products_id'],
    'sellers':['seller_id'],
    'geolocation':['geolocation_zip_code_prefix'],
    'product_category_name_translation':['product_category_name']

}

for table, pk_cols in silver_rules.items():
    df = spark.read.table(f"ecommerce_bronze.{table}")
    for pk in pk_cols:
        df.filter(col(pk).isNull())
        df.dropDuplicates(pk_cols)
        
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
            .saveAsTable(f"ecommerce_silver.{table}")
        print(f"{table}, cleaned and writen to ecommerce_silver")


In [0]:
from pyspark.sql.functions import count, sum

spark.sql("create database if not exists ecommerce_gold")

orders = spark.read.table("ecommerce_silver.orders")
order_items = spark.read.table('ecommerce_silver.order_items')
customers = spark.read.table("ecommerce_silver.customers")

order_items_agg = order_items.groupBy("order_id").agg(sum("price").alias("total_order_value"), count("*").alias("item_count"))

gold_orders = orders.join(order_items_agg,"order_id","left").join(customers, "customer_id", "left")

gold_orders.write.format('delta') \
    .mode('overwrite')\
    .option('overwriteSchema', 'true') \
     .saveAsTable('ecommerce_gold.orders')


In [0]:
%sql
select * from ecommerce_gold.orders limit 10

In [0]:
from pyspark.sql.functions import count, sum, avg

products = spark.read.table('ecommerce_silver.products')
order_items = spark.read.table('ecommerce_silver.order_items')
order_reviews = spark.read.table('ecommerce_silver.order_reviews')

product_agg = order_items.join(order_reviews, on=['order_id'], how= 'left') \
    .groupBy('product_id')\
    .agg(sum('price').alias('total_revenue'),
    count('order_id').alias('total_unit_sold'),
    avg('review_score').alias('avg_review_score'))


gold_products = products.join(product_agg, 'product_id', 'left')

gold_products.write.format('delta')\
    .mode('overwrite')\
        .option('overwriteSchema', 'true')\
            .saveAsTable('ecommerce_gold.products')

In [0]:
%sql
select * from ecommerce_gold.products limit 10
    


In [0]:
from pyspark.sql.functions import count, sum, avg

orders = spark.read.table('ecommerce_silver.orders')
order_items = spark.read.table('ecommerce_silver.order_items')
order_reviews = spark.read.table('ecommerce_silver.order_reviews')
customers = spark.read.table('ecommerce_silver.customers')

customer_agg = orders.join(order_items, 'order_id', 'left' ) \
    .join(order_reviews, 'order_id', 'left')\
        .groupBy("customer_id")\
        .agg(
            count('order_id').alias('total_order'),
            sum('price').alias('total_spent'),
            avg('review_score').alias('avg_review_score')
        )
gold_customers = customers.join(customer_agg, 'customer_id', 'left')

gold_customers.write.format('delta')\
    .mode('overwrite')\
        .option('overwriteSchema', 'true')\
            .saveAsTable('ecommerce_gold.customers')

In [0]:
%sql
select * from ecommerce_gold.customers limit 5

In [0]:
%sql
select * from ecommerce_gold.orders

In [0]:
from pyspark.sql.functions import count, sum, avg

sellers = spark.read.table('ecommerce_silver.sellers')
order_items = spark.read.table('ecommerce_silver.order_items')

sellers_agg = order_items.groupBy('seller_id')\
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_revenue')
    )
gold_seller = sellers.join(sellers_agg, "seller_id", 'left')

gold_seller.write.format('delta')\
    .mode('overwrite')\
        .option('overwriteSchema', 'true')\
            .saveAsTable('ecommerce_gold.sellers')

In [0]:
%sql
select * from ecommerce_gold.sellers limit 5